In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import yaml

sys.path.insert(0, str(Path().absolute().parent.parent.parent))

In [ ]:
import numpy as np
import pandas as pd

import matplotlib
import shutil

# Manually configure the path to the LaTeX executable from your conda env
# Note: You might need to adjust the path slightly based on your installation
latex_path = shutil.which("latex")  # This finds latex in your current PATH

if latex_path:
    matplotlib.rcParams['text.latex.preamble'] = ''
    matplotlib.rcParams['text.usetex'] = True
    matplotlib.rcParams['pgf.texsystem'] = 'pdflatex'
    matplotlib.rcParams['pgf.rcfonts'] = False
    matplotlib.rcParams['pgf.preamble'] = ''
    # Point to the TeX programs in the same directory
    matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'

    print("✅ Matplotlib's LaTeX path has been configured.")

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("notebook", font_scale=1.25)

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
# sns.set_theme(rc={'text.usetex' : True})
import numpy as np
import pandas as pd
from test.lib.files_utils import SimulationOutput, LisLoader, ISOTOPES, func

In [ ]:
lis_loader = LisLoader(Path().absolute().parent / 'data' / 'LIS_Default2020_Proton')

In [ ]:
output_files = Path().absolute().parent / 'data' / 'uncertainty' / 'outputs'
rows = []
for of in sorted(output_files.rglob('*.yaml')):
    date, _, part = of.parent.name.split('_')
    print(date, part)
    outputs = SimulationOutput(SimulationOutput.from_yaml(yaml.load(of.read_text(), Loader=yaml.CLoader)))
    fluxes = outputs.modulate(lis_loader)
    for i, (out, flux) in enumerate(zip(outputs, fluxes)):
        for j, (r, f) in enumerate(zip(*flux.rig_flux)):
            vars, evs, nbins = 0, 0, 0
            for iso, ot in out.items():
                lis = lis_loader[iso]
                in_rig, out_rig, out_dist, n_part = ot.input_rig[j:j + 1], ot.output_rig[j], ot.output_dist[j], \
                    ot.n_particles[j]

                lis_flux_en_out = func.lin_log_interpolation(lis.energy.to_rigidity(iso), lis.flux, out_rig)

                A, B, N = (lis_flux_en_out / out_rig ** 2), out_dist, n_part
                C = func.d_rig_to_en(in_rig.to_energy(iso), in_rig, iso.Z, iso.A)
                vars += C ** 2 * (np.sum(B * A ** 2) - np.sum(B * A) ** 2 / N)
                evs += C * np.sum(B * A)

                nbins = np.trim_zeros(out_dist, 'f').shape[0]

            rel_err = np.sqrt(np.sum(vars)) / np.sum(evs)

            rows.append([date, part, i, r, f, rel_err, nbins])

In [ ]:
rows[0]

In [ ]:
df = pd.DataFrame(rows, columns=['date', 'part', 'param', 'rig', 'flux', 'rel_err', 'nbins'])
df['part'] = df['part'].astype(int)
df['param'] = df['param'].astype(int)
df['rig'] = df['rig'].astype(float)
df['flux'] = df['flux'].astype(float)
df['rel_err'] = df['rel_err'].astype(float)
df['nbins'] = df['nbins'].astype(int)
df.sort_values(df.columns.to_list(), inplace=True)
df.to_csv('uncertainty.csv', index=False)
df

In [ ]:
df2 = df.copy()
df2 = df2.groupby(['date', 'part', 'rig']).agg(nbins=('nbins', 'max'), flux_mean=('flux', 'mean'),
                                               flux_std=('flux', 'std'),
                                               flux_rel=('flux', lambda x: x.std() / x.mean()),
                                               rel_err=('rel_err', lambda x: x.sample(1)),
                                               rel_err_var=('rel_err', 'var')).reset_index()

df2['flux_ratio'] = df2['flux_rel'] / df2['rel_err']
df2['flux_diff'] = (df2['flux_rel'] - df2['rel_err']) / df2['flux_rel']

df2['nbins_norm'] = df2['nbins'] / df2['nbins'].max()

df2

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

ax.set(xscale='log', yscale='log', xlabel='Rigidity', ylabel=r'$\sigma(J)/\mathrm{E}[J]$', title='Relative SD as a function of Rigidity, for 6 CRs, 30 repetitions each')

axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1)'),
    x='rig',
    y='rel_err',
    # hue='date',
    errorbar=('ci', 100),
    err_style='bars', marker='s', linestyle='', alpha=0.8,
    # marker='s',
    label='Estimated Flux Relative SD (from 1 simulation)',
    ax=ax
)

axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1)'),
    x='rig',
    y='flux_rel',
    # hue='date',
    errorbar=('ci', 100),
    err_style='bars', marker='D', linestyle='', alpha=0.8,
    # marker='D',
    label='Measured Flux Relative SD (from 30 simulations)',
    ax=ax
)

ax.grid(True, which="both", ls="-", alpha=0.5)
# ax.axhline(y=0.02, color='k', ls='--')
plt.savefig('plots/comparison.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1) '),
    x='rig',
    y='flux_ratio',
    hue='part',
    errorbar=('ci', 100),
    marker='o',
    ax=ax
)

axx.set(xscale='log', xlabel='Rigidity', ylabel=r'$\sigma(\text{Measured}) / \sigma(\text{Estimate})$', title='Ratio of Estimated and Measured Relative SD as a function of Rigidity, for 6 CRs, 6 particles')
ax.axhline(1, color='red', ls='--')

plt.legend().set_title('Particles')

plt.savefig('plots/ratio_part.pdf', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(11, 8))
axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1) '),
    x='rig',
    y='flux_ratio',
    hue='date',
    errorbar=('ci', 100),
    marker='o',
    ax=ax
)

axx.set(xscale='log', xlabel='Rigidity', ylabel=r'$\sigma(\text{Measured}) / \sigma(\text{Estimate})$', title='Ratio of Estimated and Measured Relative SD as a function of Rigidity, for 6 CRs, 6 particles')
ax.axhline(1, color='red', ls='--')

plt.legend().set_title('Date')

plt.savefig('plots/ratio_date.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1) '),
    x='rig',
    y='flux_diff',
    hue='part',
    errorbar=('ci', 100),
    marker='o',
    ax=ax
)

axx.set(xscale='log', xlabel='Rigidity', ylabel=r'$(\sigma(\text{Measured}) - \sigma(\text{Estimate})) / \sigma(\text{Measured})$', title='Relative difference of Estimated and Measured Relative SD as a function of Rigidity, for 6 CRs, 6 particles')
ax.axhline(0, color='red', ls='--')

plt.legend().set_title('Particles')

plt.savefig('plots/difference_part.pdf', bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(11, 8))
axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1) '),
    x='rig',
    y='flux_diff',
    hue='date',
    errorbar=('ci', 100),
    marker='o',
    ax=ax
)

axx.set(xscale='log', xlabel='Rigidity', ylabel=r'$(\sigma(\text{Measured}) - \sigma(\text{Estimate})) / \sigma(\text{Measured})$', title='Relative difference of Estimated and Measured Relative SD as a function of Rigidity, for 6 CRs, 6 particles')
ax.axhline(0, color='red', ls='--')

plt.legend().set_title('Date')

plt.savefig('plots/difference_date.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

ax.set(xscale='log', yscale='log', xlabel='Rigidity', ylabel=r'$\sigma(J)/\mathrm{E}[J]$', title='Relative SD as a function of Rigidity, for 6 CRs, 30 repetitions each')

axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1)'),
    x='rig',
    y='rel_err',
    hue='date',
    errorbar=('ci', 100),
    marker='s',
    ax=ax
)

axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1)'),
    x='rig',
    y='flux_rel',
    hue='date',
    errorbar=('ci', 100),
    marker='D',
    ax=ax
)

ax.grid(True, which="both", ls="-", alpha=0.5)

handles, labels = axx.get_legend_handles_labels()
date_handles = []
date_labels = []

for i, label in enumerate(labels[:len(labels)//2]):
    date_handles.append(plt.Line2D([0], [0], color=handles[i].get_color(), lw=2))
    date_labels.append(label)

experimental_handle = plt.Line2D([0], [0], marker='s', fillstyle='none', color='k', label='Estimated', markersize=8, linestyle='None')
measured_handle = plt.Line2D([0], [0], marker='D', fillstyle='none', color='k', label='Measured', markersize=8, linestyle='None')

custom_handles = date_handles + [experimental_handle, measured_handle]
custom_labels = date_labels + ['Experimental', 'Measured']

ax.legend(custom_handles, custom_labels, title='Date and Type', loc='best')

plt.savefig('plots/comparison_date.pdf', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))

ax.set(xscale='log', yscale='log', xlabel='Rigidity', ylabel=r'$\sigma(J)/\mathrm{E}[J]$', title='Relative SD as a function of Rigidity, for 6 CRs, 30 repetitions each')

axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1)'),
    x='rig',
    y='rel_err',
    hue='part',
    errorbar=('ci', 100),
    marker='s',
    ax=ax
)

axx = sns.lineplot(
    df2.query('(rig<1001) & (part > 1)'),
    x='rig',
    y='flux_rel',
    hue='part',
    errorbar=('ci', 100),
    marker='D',
    ax=ax
)

ax.grid(True, which="both", ls="-", alpha=0.5)

handles, labels = axx.get_legend_handles_labels()
date_handles = []
date_labels = []

for i, label in enumerate(labels[:len(labels)//2]):
    date_handles.append(plt.Line2D([0], [0], color=handles[i].get_color(), lw=2))
    date_labels.append(label)

experimental_handle = plt.Line2D([0], [0], marker='s', fillstyle='none', color='k', label='Estimated', markersize=8, linestyle='None')
measured_handle = plt.Line2D([0], [0], marker='D', fillstyle='none', color='k', label='Measured', markersize=8, linestyle='None')

custom_handles = date_handles + [experimental_handle, measured_handle]
custom_labels = date_labels + ['Experimental', 'Measured']

ax.legend(custom_handles, custom_labels, title='Particles and Type', loc='best')

plt.savefig('plots/comparison_part.pdf', bbox_inches='tight')
plt.show()

In [ ]:
def threshold(t):
    return np.sqrt(2) * (1.29 * t ** (-0.85)) / (t ** -0.85 + 1.5)


fig, ax = plt.subplots(figsize=(10, 9))
axx = sns.lineplot(
    df2.query('(rig<1001)'),
    x='rig',
    y='flux_ratio',
    hue='date',
    # errorbar=('ci', .05),
    err_style='bars', marker='o', linestyle='',
    ax=ax
)
sns.lineplot(
    x=df2.query('(rig<101)')['rig'],
    y=threshold(df2.query('(rig<101)')['rig']),
    color='k',
    label='old threshold',
    ax=ax,
)

sns.lineplot(
    df2,
    x='rig',
    y='nbins_norm',
    hue='date',
)
axx.set(xscale='log', yscale='log', xlabel='Energy', ylabel=r'(sigma sim) / (sigma pois)')
# axx.axhline(2*10**-2)

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 9))
#
# ax.set(xscale='log', xlabel='Rigidity', ylabel=r'$\sigma$')
#
# axx = sns.lineplot(
#     df2.query('(rig<1001) & (part > 1)'),
#     x='rig',
#     y='bias',
#     hue='part',
#     errorbar=('ci', 95),
#     # err_style='bars', marker='o', linestyle='',
#     marker='o',
#     ax=ax
# )
# # axx = sns.lineplot(
# #     df2.query('(rig<1001) & (part > 16000)'),
# #     x='rig',
# #     y='nbins_norm',
# #     # hue='date',
# #     errorbar=('ci', 95),
# #     # err_style='bars', marker='o', linestyle='',
# #     marker='o',
# #     ax=ax
# # )
#
# plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 9))
axx = sns.boxplot(
    df2.query('(rig<1001)'),
    x='rig',
    y='flux_ratio_norm',
    hue='part',
    # errorbar=('ci', .05),
    # err_style='bars', marker='o', linestyle='',
    palette='muted',
    ax=ax
)

axx.set(xlabel='Energy', ylabel=r'(sigma sim) / (sigma pois)')
plt.xticks(rotation=90)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
axx = sns.lineplot(
    df2.query('(rig<1001)'),
    x='rig',
    y='avg_ratio',
    hue='date',
    # errorbar=('ci', .05),
    err_style='bars', marker='o', linestyle='',
    ax=ax
)

axx.set(xscale='log', yscale='log', xlabel='Energy', ylabel=r'(sigma sim) / (sigma pois)')
# axx.axhline(2*10**-2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
axx = sns.lineplot(
    df2.query('(rig<1001)'),
    x='rig',
    y='avg_std',
    hue='date',
    # errorbar=('ci', .05),
    err_style='bars', marker='o', linestyle='',
    ax=ax
)

axx.set(xscale='log', yscale='log', xlabel='Energy', ylabel=r'(sigma sim) / (sigma pois)')
# axx.axhline(2*10**-2)